# RQ3 - Smart charging driven by demand forecasts

**RQ3.** By what percentage can accurate charging-demand forecasting reduce the risk of local grid overload (peak load) compared with uncontrolled charging, and what does this suggest for a smart-charging strategy?

**Design.**
1. *Uncontrolled charging*: every session charges at constant power from plug-in to the recorded end of charging (this is how the hourly series `y` is built).
2. *Grid limit (assumption)*: the dataset has no transformer rating, so the local limit of a site is defined as the 95th percentile (P95; P90 as sensitivity) of the **Train** hourly uncontrolled load.
3. *Smart charging*: a causal, energy-preserving, forecast-driven **cap scheduler**. For day *d* the cap is `C_d = kappa x (forecast peak of day d)`, using the day-ahead (H=24) forecast of the 24 hours of day d. Hour by hour, plugged-in sessions are served least-laxity-first so that the site load stays below `C_d`; a session whose remaining energy would become infeasible before departure is forced to charge at the required power even if the cap is exceeded (energy and deadlines are always respected). `kappa` is tuned on Val for every forecaster and applied unchanged on Test.
4. *Forecasters*: every model supplied in the embedded forecast table, plus an **oracle** (true daily peak) and a **last-week peak** rule (no model).
5. *Outputs*: % reduction versus uncontrolled in mean daily peak, maximum peak, overload hours and overload energy (95% CI by week-block bootstrap), cost of smart charging (energy shifted, mean delay) and the link between forecast accuracy and peak reduction.

In [ ]:
MODEL_TAG, MODEL_NAME = 'rq3_smart_charging', 'RQ3 smart charging'

## 0. Shared benchmark configuration and data (identical to the RQ2 notebooks)

In [ ]:
# ====================== SHARED CONFIG (identical in every RQ2/RQ3 notebook) ======================
import os, sys, glob, json, random, warnings, subprocess, time, hashlib, platform, base64, gzip, io
warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from numpy.lib.stride_tricks import sliding_window_view
from pandas.tseries.holiday import USFederalHolidayCalendar
from sklearn.metrics import mean_absolute_error, mean_squared_error
pd.set_option('display.width', 250, 'display.max_columns', None, 'display.max_rows', 500)

FAST_MODE   = False        # True = smoke test only
SEED        = 42
SITES       = [1, 2]       # one hourly series (and one model) per site; site 19 is dropped (too sparse)
HORIZONS    = [1, 24]      # H=1 next hour, H=24 one day ahead (direct strategy)

# ---- Split requested by the author: Train -> 31/10/2019, Val -> 15/01/2020, Test = 16/01/2020 -> end of data ----
TRAIN_END   = '2019-10-31 23:00'
VAL_END     = '2020-01-15 23:00'
# Two training protocols:
#   run1 : fit on Train, early-stop / select on Val, evaluate on Test        (train - val - test)
#   run2 : refit on Train+Val with the settings chosen in run1, evaluate on Test   (train(train+val) - test)

MAX_LOOKBACK = 168         # longest history window (hours) any model may read
WARMUP_H     = 24 * 28     # first 4 weeks of each series are never targets (672 h of history needed by seasonal_mean_4w)
GAP_DAYS     = 7           # >= 7 consecutive hours-days with zero energy = data outage, not demand
POST_GAP_H   = 24 * 28     # after every outage the next 4 weeks are not targets (lags would read the artificial zeros)
MAPE_MIN     = 0.5         # kWh; MAPE/sMAPE only on hours with at least this load
N_BOOT       = 1000
BLOCK_H      = 24 * 7
if FAST_MODE: N_BOOT = 100
random.seed(SEED); np.random.seed(SEED)
plt.rcParams.update({'figure.dpi': 110, 'axes.grid': True, 'grid.alpha': .3,
                     'axes.spines.top': False, 'axes.spines.right': False})

# ---- The 16 common features (CFV) used by EVERY forecasting model ----
FEATS_COMMON = ['lag_H', 'lag_D', 'lag_168', 'roll_mean_24', 'roll_max_24', 'seasonal_mean_4w',
                'n_plugged_lagH', 'n_charging_lagH', 'n_arrivals_lagH', 'charging_share_lagH',
                'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
assert len(FEATS_COMMON) == 16
FEATS_HASH = hashlib.md5('|'.join(FEATS_COMMON).encode()).hexdigest()[:10]
CAL = ['hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'is_weekend', 'is_holiday']
HIST_COLS = ['y_s', 'plug_s', 'chg_s', 'arr_s']          # 4 dynamic channels for recurrent nets

In [ ]:
# ---------- Output folder + data file ----------
OUT_DIR = '/kaggle/working' if os.path.exists('/kaggle/working') else './outputs_' + MODEL_TAG
FIG_DIR = os.path.join(OUT_DIR, 'figures'); os.makedirs(FIG_DIR, exist_ok=True)
def savefig(name):
    plt.savefig(os.path.join(FIG_DIR, f'{MODEL_TAG}_{name}.png'), dpi=140, bbox_inches='tight'); plt.show()
def resolve_data():
    pats = ['/kaggle/input/**/acn_data*.xlsx', '/kaggle/input/**/acn*.xlsx', './acn*.xlsx', '/mnt/user-data/uploads/**/acn*.xlsx']
    for p in pats:
        hits = sorted(glob.glob(p, recursive=True))
        if hits: return hits[0]
    raise FileNotFoundError('acn*.xlsx not found. Add the ACN dataset as notebook input.')
DATA_PATH = resolve_data()
print('Model:', MODEL_NAME, '| data:', DATA_PATH, '| python', sys.version.split()[0])

# ---------- Read + clean sessions ----------
try: raw = pd.read_excel(DATA_PATH, sheet_name='full_data')
except ValueError: raw = pd.read_excel(DATA_PATH)
for c in ['connectionTime', 'disconnectTime', 'doneChargingTime']: raw[c] = pd.to_datetime(raw[c])
sess = raw.drop_duplicates(['stationID', 'connectionTime']).copy()
sess['done'] = sess['doneChargingTime'].clip(upper=sess['disconnectTime'])
sess = sess[(sess.kWhDelivered > 0) & (sess.disconnectTime > sess.connectionTime)]
sess = sess[sess.siteID.isin(SITES)].reset_index(drop=True)
print(f'sessions after cleaning (sites {SITES}): {len(sess):,} / raw {len(raw):,}')

# ---------- Hourly load series per site ----------
def spread_energy(s):
    c = s.connectionTime.values.astype('datetime64[s]'); e = s.done.values.astype('datetime64[s]')
    e = np.maximum(e, c + np.timedelta64(60, 's'))
    c0 = c.astype('datetime64[h]'); e0 = (e - np.timedelta64(1, 's')).astype('datetime64[h]')
    n = ((e0 - c0) // np.timedelta64(1, 'h')).astype(int) + 1
    idx = np.repeat(np.arange(len(s)), n)
    k = np.arange(n.sum()) - np.repeat(np.cumsum(n) - n, n)
    hs = c0[idx] + k.astype('timedelta64[h]'); he = hs + np.timedelta64(1, 'h')
    ov = (np.minimum(e[idx], he) - np.maximum(c[idx], hs)) / np.timedelta64(1, 's')
    tot = ((e - c) / np.timedelta64(1, 's'))[idx]
    return pd.Series(s.kWhDelivered.values[idx] * ov / tot, index=pd.DatetimeIndex(hs)).groupby(level=0).sum()

def hourly_site(s):
    y = spread_energy(s)
    grid = pd.date_range(s.connectionTime.min().floor('h'), s.done.max().ceil('h'), freq='h', name='time')
    out = pd.DataFrame(index=grid); out['y'] = y.reindex(grid).fillna(0.0)
    ends = (grid + pd.Timedelta(hours=1)).values
    conn = np.sort(s.connectionTime.values); disc = np.sort(s.disconnectTime.values); done = np.sort(s.done.values)
    out['n_plugged_end']  = np.searchsorted(conn, ends, 'right') - np.searchsorted(disc, ends, 'right')
    out['n_charging_end'] = np.searchsorted(conn, ends, 'right') - np.searchsorted(done, ends, 'right')
    out['n_arrivals'] = pd.Series(1, index=s.connectionTime.dt.floor('h')).groupby(level=0).sum().reindex(grid).fillna(0).values
    return out

hol = USFederalHolidayCalendar().holidays('2018-01-01', '2021-12-31')
def add_calendar(Hs):
    ix = Hs.index
    Hs['hour_sin'] = np.sin(2*np.pi*ix.hour/24);     Hs['hour_cos'] = np.cos(2*np.pi*ix.hour/24)
    Hs['dow_sin']  = np.sin(2*np.pi*ix.dayofweek/7); Hs['dow_cos']  = np.cos(2*np.pi*ix.dayofweek/7)
    Hs['is_weekend'] = (ix.dayofweek >= 5).astype(float)
    Hs['is_holiday'] = ix.normalize().isin(hol).astype(float)     # US federal calendar, NOT the file's is_holiday column
    return Hs

T_END, V_END = pd.Timestamp(TRAIN_END), pd.Timestamp(VAL_END)
HOURLY, GAP_RUNS, SCALE, THR90, MASE_SCALE = {}, {}, {}, {}, {}
for sid in SITES:
    s = sess[sess.siteID == sid]
    Hs = hourly_site(s)
    assert abs(Hs.y.sum() - s.kWhDelivered.sum()) < 1e-3, 'kWh not conserved'
    inactive = (Hs.y == 0); grp = (inactive != inactive.shift()).cumsum()
    Hs['in_gap'] = inactive & (inactive.groupby(grp).transform('size') >= GAP_DAYS * 24)
    _g = (Hs.in_gap != Hs.in_gap.shift()).cumsum()
    GAP_RUNS[sid] = [(g.index.min(), g.index.max()) for _, g in Hs[Hs.in_gap].groupby(_g[Hs.in_gap])]
    post = np.zeros(len(Hs), bool)
    for a, b in GAP_RUNS[sid]:
        j = Hs.index.get_loc(b); post[j + 1: j + 1 + POST_GAP_H] = True
    Hs['post_gap'] = post
    Hs['valid'] = (~Hs.in_gap.values) & (~post) & (np.arange(len(Hs)) >= WARMUP_H)   # identical target hours for every model
    Hs = add_calendar(Hs)
    tr_ok = (Hs.index <= T_END) & Hs.valid.values
    SCALE[sid] = {c: max(float(np.quantile(Hs.loc[tr_ok, c], .99)), 1e-6) for c in ['y', 'n_plugged_end', 'n_charging_end', 'n_arrivals']}
    Hs['y_s'] = Hs.y / SCALE[sid]['y']; Hs['plug_s'] = Hs.n_plugged_end / SCALE[sid]['n_plugged_end']
    Hs['chg_s'] = Hs.n_charging_end / SCALE[sid]['n_charging_end']; Hs['arr_s'] = Hs.n_arrivals / SCALE[sid]['n_arrivals']
    THR90[sid] = float(np.quantile(Hs.loc[tr_ok, 'y'], .90))
    d24 = (Hs.y - Hs.y.shift(24)).abs()[tr_ok & Hs.valid.shift(24, fill_value=False).values].dropna()
    MASE_SCALE[sid] = float(d24.mean())
    HOURLY[sid] = Hs
    print(f'site {sid}: {len(Hs):,} h | {Hs.index.min()} -> {Hs.index.max()} | valid target hours {int(Hs.valid.sum()):,} | P90(train)={THR90[sid]:.2f} kWh')
    for a, b in GAP_RUNS[sid]: print(f'    outage {a} -> {b} ({(b-a).days+1} days)')

# ---------- Split (time based) ----------
def split_of(t):
    t = np.asarray(t)
    return np.select([t <= T_END, t <= V_END], ['train', 'val'], default='test')
def target_index(site): return np.flatnonzero(HOURLY[site].valid.values)

_rows = []
for sid in SITES:
    ti = target_index(sid); spl = split_of(HOURLY[sid].index.values[ti])
    for nm in ['train', 'val', 'test']:
        m = spl == nm
        _rows.append(dict(site=sid, split=nm, n_target_hours=int(m.sum()), first=HOURLY[sid].index[ti][m].min(), last=HOURLY[sid].index[ti][m].max()))
SPLIT_TBL = pd.DataFrame(_rows).set_index(['site', 'split']); print(SPLIT_TBL.to_string())
BENCH_FINGERPRINT = hashlib.md5('|'.join(map(str, [SITES, HORIZONS, TRAIN_END, VAL_END, MAX_LOOKBACK, WARMUP_H, GAP_DAYS,
                                  POST_GAP_H, MAPE_MIN, FEATS_HASH, SEED])).encode()).hexdigest()[:12]
print('BENCHMARK FINGERPRINT =', BENCH_FINGERPRINT, '| features hash =', FEATS_HASH)

# ---------- 16 common features ----------
def build_common_features(Hs, h):
    """16 columns; every value uses information up to hour t-h only, or is a deterministic function of the calendar."""
    y = Hs.y; D = 24 if h < 24 else 48; f = pd.DataFrame(index=Hs.index)
    f['lag_H'] = y.shift(h); f['lag_D'] = y.shift(D); f['lag_168'] = y.shift(168)
    ys = y.shift(h)
    f['roll_mean_24'] = ys.rolling(24).mean(); f['roll_max_24'] = ys.rolling(24).max()
    f['seasonal_mean_4w'] = pd.concat([y.shift(168 * k) for k in range(1, 5)], axis=1).mean(axis=1)
    f['n_plugged_lagH'] = Hs.n_plugged_end.shift(h); f['n_charging_lagH'] = Hs.n_charging_end.shift(h)
    f['n_arrivals_lagH'] = Hs.n_arrivals.shift(h)
    f['charging_share_lagH'] = (f.n_charging_lagH / f.n_plugged_lagH.replace(0, np.nan)).fillna(0.0)
    for c in CAL: f[c] = Hs[c]
    return f[FEATS_COMMON]

def scale_features(X, site):
    """Same definition, scaled for neural nets (load features / P99(y); count features / P99(count); calendar unchanged)."""
    Z = X.copy(); sy = SCALE[site]['y']
    for c in ['lag_H', 'lag_D', 'lag_168', 'roll_mean_24', 'roll_max_24', 'seasonal_mean_4w']: Z[c] = Z[c] / sy
    Z['n_plugged_lagH'] /= SCALE[site]['n_plugged_end']; Z['n_charging_lagH'] /= SCALE[site]['n_charging_end']
    Z['n_arrivals_lagH'] /= SCALE[site]['n_arrivals']
    return Z.values.astype('float32')

def make_data(site, h, sequences=False):
    Hs = HOURLY[site]; tgt = target_index(site); spl = split_of(Hs.index[tgt]); L = MAX_LOOKBACK
    f = build_common_features(Hs, h)
    if sequences:
        S = Hs[HIST_COLS].values.astype('float32'); Wv = sliding_window_view(S, L, axis=0).swapaxes(1, 2)
    def pack(sel):
        X = f.iloc[sel]
        assert not X.isna().any().any(), 'NaN in features'
        d = dict(idx=sel, X=X, Xs=scale_features(X, site), y=Hs.y.values[sel], y_s=Hs.y_s.values[sel].astype('float32'))
        if sequences: d['hist'] = Wv[sel - h - L + 1].copy()
        return d
    D = {n: pack(tgt[spl == n]) for n in ['train', 'val', 'test']}
    D['dev'] = pack(tgt[np.isin(spl, ['train', 'val'])])
    return D

# ---------- Evaluation library ----------
def pred_frame(site, h, run, split, idx, pred):
    Hs = HOURLY[site]
    return pd.DataFrame({'time': Hs.index[idx], 'site': site, 'horizon': h, 'run': run, 'split': split,
                         'actual': Hs.y.values[idx], 'pred': np.clip(np.asarray(pred, float), 0, None)})

def hourly_stats(d, thr):
    a, p = d.actual.values, d.pred.values; e = p - a; m = a >= MAPE_MIN
    ape = np.where(m, np.abs(e) / np.where(m, a, 1.0), 0.0); sape = np.where(m, 2 * np.abs(e) / np.maximum(a + p, 1e-9), 0.0)
    return np.column_stack([np.ones(len(a)), a, a**2, p, p**2, a*p, np.abs(e), e, e**2, m.astype(float), ape, sape,
                            ((p >= thr) & (a >= thr)).astype(float), (p >= thr).astype(float), (a >= thr).astype(float)])
def make_hourly_fn(scale):
    def fn(s):
        n, sa, saa, sp, spp, sap, sae, se, see, nm, sape, ssape, tp, npp, nap = [s[:, i] for i in range(15)]
        with np.errstate(divide='ignore', invalid='ignore'):
            sst = saa - sa**2 / n; cv = sap - sa * sp / n; vp = spp - sp**2 / n
            return {'MAE': sae / n, 'RMSE': np.sqrt(see / n), 'R2': 1 - see / sst, 'MAPE_%': 100 * sape / nm,
                    'sMAPE_%': 100 * ssape / nm, 'WAPE_%': 100 * sae / sa, 'MASE': sae / n / scale, 'Bias_kWh': se / n,
                    'Pearson_r': cv / np.sqrt(sst * vp), 'P90_precision': tp / npp, 'P90_recall': tp / nap, 'P90_F1': 2 * tp / (npp + nap)}
    return fn
def daily_peaks(d):
    g = d.groupby(d.index.normalize()); ia, ip = g.actual.idxmax(), g.pred.idxmax()
    pk = pd.DataFrame({'act_peak': g.actual.max(), 'pred_peak': g.pred.max(), 'act_h': ia.dt.hour, 'pred_h': ip.dt.hour})
    return pk[pk.act_peak > 0]
def peak_stats(pk):
    a = pk.act_peak.values; e = (pk.pred_peak - pk.act_peak).values
    dh = (pk.pred_h - pk.act_h).abs().values; dh = np.minimum(dh, 24 - dh)
    return np.column_stack([np.ones(len(a)), a, a**2, np.abs(e), e, e**2, (dh <= 1).astype(float)])
def peak_fn(s):
    n, sa, saa, sae, se, see, hit = [s[:, i] for i in range(7)]
    with np.errstate(divide='ignore', invalid='ignore'):
        return {'Peak_MAE': sae / n, 'Peak_WAPE_%': 100 * sae / sa, 'Peak_Bias_%': 100 * se / sa, 'PeakHour_hit_pm1h_%': 100 * hit / n}
def _boot(S, fn, block, rng):
    n = len(S)
    if n / block < 8: block = max(1, block // 7)
    B = np.add.reduceat(S, np.arange(0, n, block), axis=0)
    idx = rng.integers(0, len(B), size=(N_BOOT, len(B)))
    boot = fn(B[idx].sum(axis=1)); point = fn(S.sum(axis=0, keepdims=True)); out = {}
    for k in point:
        b = boot[k][np.isfinite(boot[k])]; ok = len(b) > 2
        out[k] = (float(point[k][0]), float(b.std(ddof=1)) if ok else np.nan)
    return out
def evaluate_all(PRED, seed=SEED):
    rng = np.random.default_rng(seed); rows = []
    for (run, site, h), g in PRED.groupby(['run', 'site', 'horizon']):
        g = g.sort_values('time').set_index('time'); hfn = make_hourly_fn(MASE_SCALE[site])
        for name, d in {'Train': g[g.split == 'train'], 'Val': g[g.split == 'val'], 'Test': g[g.split == 'test']}.items():
            if len(d) < 48: continue
            res = _boot(hourly_stats(d, THR90[site]), hfn, BLOCK_H, rng); pk = daily_peaks(d)
            resp = _boot(peak_stats(pk), peak_fn, 7, rng) if len(pk) >= 8 else {}
            for src in (res, resp):
                for k, v in src.items():
                    rows.append(dict(run=run, site=site, horizon=h, group=name, metric=k, value=v[0], SE=v[1], n_hours=len(d)))
    return pd.DataFrame(rows)
MAIN = ['MAE', 'RMSE', 'R2', 'WAPE_%', 'MASE', 'Bias_kWh', 'P90_F1', 'Peak_MAE']

def standard_report(PRED, extra_json=None):
    """Save predictions + metrics + standard figures; print compact Test tables for both runs."""
    PRED = PRED.sort_values(['run', 'site', 'horizon', 'time']).reset_index(drop=True)
    PRED.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_predictions.csv'), index=False)
    RES = evaluate_all(PRED); RES.insert(0, 'model', MODEL_NAME)
    RES.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_results_long.csv'), index=False)
    T = RES[RES.group.isin(['Val', 'Test']) & RES.metric.isin(MAIN)]
    W = T.pivot_table(index=['run', 'site', 'horizon', 'group'], columns='metric', values='value')[[m for m in MAIN if m in T.metric.unique()]]
    W.to_csv(os.path.join(OUT_DIR, f'{MODEL_TAG}_results_wide.csv'))
    print(f'===== {MODEL_NAME}: Val / Test metrics =====\n', W.round(4).to_string())
    # ---- figures ----
    runs = [r for r in ['run2', 'run1'] if r in PRED.run.unique()]
    r0 = runs[0]; fig, axs = plt.subplots(len(SITES), len(HORIZONS), figsize=(7 * len(HORIZONS), 3.4 * len(SITES)), squeeze=False)
    for i, site in enumerate(SITES):
        for j, h in enumerate(HORIZONS):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')].set_index('time')
            g = g[g.index >= g.index.max() - pd.Timedelta(days=14)]
            axs[i, j].plot(g.index, g.actual, color='#444', lw=1.1, label='Actual'); axs[i, j].plot(g.index, g.pred, color='#D55E00', lw=1.1, label='Forecast')
            axs[i, j].set(title=f'{MODEL_NAME} | site {site} | H={h}: last 14 test days', ylabel='kWh per hour'); axs[i, j].legend(fontsize=7)
            axs[i, j].tick_params(axis='x', rotation=20, labelsize=7)
    plt.tight_layout(); savefig('F1_forecast_window')
    fig, axs = plt.subplots(len(SITES), len(HORIZONS), figsize=(4.2 * len(HORIZONS), 3.8 * len(SITES)), squeeze=False)
    for i, site in enumerate(SITES):
        for j, h in enumerate(HORIZONS):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')]
            axs[i, j].scatter(g.actual, g.pred, s=4, alpha=.25, color='#0072B2'); m = max(g.actual.max(), g.pred.max())
            axs[i, j].plot([0, m], [0, m], 'k--', lw=.8); axs[i, j].set(title=f'site {site} | H={h}', xlabel='Actual kWh', ylabel='Forecast kWh')
    plt.suptitle(f'{MODEL_NAME}: actual vs forecast (Test)', y=1.02); plt.tight_layout(); savefig('F2_scatter')
    fig, axs = plt.subplots(1, len(SITES), figsize=(5.5 * len(SITES), 3.4), squeeze=False)
    for i, site in enumerate(SITES):
        for h, c in zip(HORIZONS, ['#009E73', '#CC79A7']):
            g = PRED[(PRED.run == r0) & (PRED.site == site) & (PRED.horizon == h) & (PRED.split == 'test')]
            e = (g.actual - g.pred).abs().groupby(g.time.dt.hour).mean(); axs[0, i].plot(e.index, e.values, 'o-', ms=3, color=c, label=f'H={h}')
        axs[0, i].set(title=f'site {site}: MAE by hour of day (Test)', xlabel='Hour', ylabel='MAE (kWh)'); axs[0, i].legend(fontsize=7)
    plt.tight_layout(); savefig('F3_error_by_hour')
    if extra_json is not None:
        S = {'model': MODEL_NAME, 'tag': MODEL_TAG, 'fingerprint': BENCH_FINGERPRINT, 'features': FEATS_COMMON,
             'split': {'TRAIN_END': TRAIN_END, 'VAL_END': VAL_END, 'TEST': '2020-01-16 -> end of data'}, **extra_json}
        with open(os.path.join(OUT_DIR, f'{MODEL_TAG}_best_hyperparameters.json'), 'w') as f: json.dump(S, f, indent=2, default=str)
    return RES, W
print('core ready')

## 1. Embedded forecasts from the RQ2 models

In [ ]:
import io, gzip, base64, glob
PRED_B64 = ''
_hits = glob.glob('/kaggle/input/**/rq2_combined_predictions.csv', recursive=True)
if _hits:
    PRED_CSV = _hits[0]; print('forecasts read from the RQ2 aggregate notebook:', PRED_CSV)
else:
    PRED_CSV = None; print('rq2_combined_predictions.csv not found; using the embedded forecast table:', len(PRED_B64)//1024, 'kB')

## 2. Smart-charging simulator

In [ ]:
# ---------- RQ3 settings ----------
RQ3_FAST   = False
KAPPAS     = [0.5, 0.6, 0.7, 0.8, 0.9, 1.0] if not RQ3_FAST else [0.7, 1.0]
KAPPA_FIX  = 0.8                  # fixed-kappa sensitivity (same for every forecaster)
P_MIN_KW   = 6.6                  # minimum per-session maximum power (J1772 level 2)
N_BOOT3    = 1000 if not RQ3_FAST else 100
LIMIT_Q    = {'P95': 0.95, 'P90': 0.90}   # grid limit = quantile of Train uncontrolled hourly load (assumption)
SPILL_H    = 72

# ---------- forecasts ----------
if PRED_CSV is not None: PRED_ALL = pd.read_csv(PRED_CSV, parse_dates=['time'])
elif PRED_B64: PRED_ALL = pd.read_csv(io.BytesIO(gzip.decompress(base64.b64decode(PRED_B64))), parse_dates=['time'])
else: raise FileNotFoundError('Add the output of the rq2-aggregate notebook as an input (file rq2_combined_predictions.csv).')
PRED_ALL = PRED_ALL[(PRED_ALL.horizon == 24) & PRED_ALL.split.isin(['val', 'test'])]
MODELS = sorted(PRED_ALL.model.unique())
if RQ3_FAST: MODELS = MODELS[:2]
FC = {}
for (m, site, run), g in PRED_ALL.groupby(['model', 'site', 'run']):
    FC[(m, int(site), run)] = g.drop_duplicates('time').set_index('time').pred
print('forecasters supplied:', MODELS)

LIMIT = {sid: {k: float(np.quantile(HOURLY[sid].loc[(HOURLY[sid].index <= T_END) & HOURLY[sid].valid.values, 'y'], q))
               for k, q in LIMIT_Q.items()} for sid in SITES}
print('grid limit (kWh/h):', LIMIT)

LAST_DATE = max(HOURLY[s].index.max() for s in SITES).normalize()
PERIODS = {'val': (pd.Timestamp(T_END.date()) + pd.Timedelta(days=1), pd.Timestamp(V_END.date())),
           'test': (pd.Timestamp(V_END.date()) + pd.Timedelta(days=1), LAST_DATE)}

def eval_days(site, t0, t1):
    """Days on which all 24 hours (and the same weekday one week earlier) are valid target hours: common to every forecaster."""
    v = HOURLY[site].valid; ok = []
    for d in pd.date_range(t0, t1, freq='D'):
        h = pd.date_range(d, periods=24, freq='h'); hp = h - pd.Timedelta(days=7)
        if v.reindex(h, fill_value=False).all() and v.reindex(hp, fill_value=False).all(): ok.append(d)
    return pd.DatetimeIndex(ok)

# ---------- session preparation ----------
def prep_sessions(site, t0, t1):
    s = sess[(sess.siteID == site) & (sess.connectionTime >= t0) & (sess.connectionTime < t1 + pd.Timedelta(days=1))].copy()
    base = t0
    c = ((s.connectionTime - base) / pd.Timedelta(hours=1)).values
    d = ((s.disconnectTime - base) / pd.Timedelta(hours=1)).values
    dn = ((s.done - base) / pd.Timedelta(hours=1)).values
    dn = np.maximum(dn, c + 1 / 60)
    E = s.kWhDelivered.values
    p_unc = E / (dn - c)
    r = np.maximum(P_MIN_KW, p_unc)
    order = np.argsort(c); 
    return dict(c=c[order], d=d[order], dn=dn[order], E=E[order], r=r[order], frame=s.iloc[order], base=base)

def uncontrolled_load(S, nH):
    sf = S['frame']; y = spread_energy(sf)
    grid = pd.date_range(S['base'], periods=nH + SPILL_H, freq='h')
    return y.reindex(grid).fillna(0.0).values

def simulate(S, nH, capH):
    """Least-laxity-first cap scheduler. capH: kWh per hour cap for every hour of the grid (np.inf = no cap)."""
    c, d, E, r = S['c'], S['d'], S['E'], S['r']; n = len(c); rem = E.copy()
    cent = np.zeros(n); load = np.zeros(nH + SPILL_H); viol = np.zeros(n, bool)
    arr_h = np.floor(c).astype(int); ptr = 0
    act = np.empty(0, dtype=int)
    for tau in range(nH + SPILL_H):
        new = []
        while ptr < n and arr_h[ptr] <= tau: new.append(ptr); ptr += 1
        if new: act = np.concatenate([act, np.array(new, dtype=int)])
        if act.size == 0:
            if ptr >= n: break
            continue
        ra = rem[act]; da = d[act]; rr = r[act]; ca = c[act]
        avail = np.clip(np.minimum(da, tau + 1) - np.maximum(ca, tau), 0, 1)
        mx = np.minimum(ra, rr * avail)
        fut = rr * np.maximum(0.0, da - (tau + 1))
        forced = np.clip(ra - fut, 0, None)
        bad = forced > mx + 1e-9
        if bad.any(): viol[act[bad]] = True
        forced = np.minimum(forced, mx)
        extra = mx - forced; cap = capH[tau] if tau < len(capH) else np.inf
        if np.isfinite(cap):
            left = max(0.0, cap - forced.sum())
            lax = (fut + mx - ra) / rr
            o = np.argsort(lax, kind='stable'); cs = np.cumsum(extra[o]) - extra[o]
            al = np.empty_like(extra); al[o] = np.clip(left - cs, 0, extra[o])
        else:
            al = extra
        tot = forced + al
        rem[act] = ra - tot; load[tau] = tot.sum(); cent[act] += tot * (tau + 0.5)
        keep = (rem[act] > 1e-9) & (da > tau + 1)
        viol[act[(~keep) & (rem[act] > 1e-6)]] = True
        act = act[keep]
    viol |= rem > 1e-6
    return dict(load=load, rem=rem, cent=cent, viol=viol)

def caps_from_peaks(peaks, days, nH, kappa):
    capH = np.full(nH + SPILL_H, np.inf); d0 = days[0]
    for d, pk in peaks.items():
        i = int((d - d0) / pd.Timedelta(days=1)) * 24
        if 0 <= i < nH and np.isfinite(pk): capH[i:i + 24] = max(kappa * pk, 1e-6)
    return capH

def forecast_peaks(site, run, model, edays):
    ser = FC.get((model, site, run)); out = {}
    if ser is None: return out
    for d in edays:
        v = ser.reindex(pd.date_range(d, periods=24, freq='h'))
        if v.notna().all(): out[d] = float(v.max())
    return out
def lastweek_peaks(site, edays):
    y = HOURLY[site].y; return {d: float(y.reindex(pd.date_range(d - pd.Timedelta(days=7), periods=24, freq='h')).max()) for d in edays}

def day_stats(u, s, nH, edays, days, site):
    """Per-day statistics on evaluation days."""
    d0 = days[0]; di = np.array([int((d - d0) / pd.Timedelta(days=1)) for d in edays])
    U = u[:nH].reshape(-1, 24)[di]; Sx = s[:nH].reshape(-1, 24)[di]
    out = dict(pk_u=U.max(1), pk_s=Sx.max(1), week=di // 7)
    for k, L in LIMIT[site].items():
        out[f'oe_u_{k}'] = np.clip(U - L, 0, None).sum(1); out[f'oe_s_{k}'] = np.clip(Sx - L, 0, None).sum(1)
        out[f'oh_u_{k}'] = (U > L).sum(1).astype(float); out[f'oh_s_{k}'] = (Sx > L).sum(1).astype(float)
    return out
def red(a, b): return 100.0 * (1 - b / a) if a > 0 else np.nan
def summarize(st, rng):
    keep = st['pk_u'] > 0; st = {k: v[keep] for k, v in st.items()}
    res = dict(n_days=int(keep.sum()), mean_peak_u=float(st['pk_u'].mean()), mean_peak_s=float(st['pk_s'].mean()),
               max_peak_u=float(st['pk_u'].max()), max_peak_s=float(st['pk_s'].max()))
    res['red_mean_peak_%'] = red(res['mean_peak_u'], res['mean_peak_s']); res['red_max_peak_%'] = red(res['max_peak_u'], res['max_peak_s'])
    for k in LIMIT_Q:
        res[f'overload_h_u_{k}'] = float(st[f'oh_u_{k}'].sum()); res[f'overload_h_s_{k}'] = float(st[f'oh_s_{k}'].sum())
        res[f'overload_kWh_u_{k}'] = float(st[f'oe_u_{k}'].sum()); res[f'overload_kWh_s_{k}'] = float(st[f'oe_s_{k}'].sum())
        res[f'red_overload_h_{k}_%'] = red(res[f'overload_h_u_{k}'], res[f'overload_h_s_{k}'])
        res[f'red_overload_kWh_{k}_%'] = red(res[f'overload_kWh_u_{k}'], res[f'overload_kWh_s_{k}'])
    # week-block bootstrap
    wk = st['week']; uw = np.unique(wk); idx = {w: np.flatnonzero(wk == w) for w in uw}
    keys = ['pk_u', 'pk_s'] + [f'{a}_{b}_{k}' for a in ('oe', 'oh') for b in ('u', 's') for k in LIMIT_Q]
    agg = {k: np.array([st[k][idx[w]].sum() for w in uw]) for k in keys}; nd = np.array([len(idx[w]) for w in uw])
    bs = {'red_mean_peak_%': [], 'red_overload_kWh_P95_%': [], 'red_overload_h_P95_%': []}
    for _ in range(N_BOOT3):
        b = rng.integers(0, len(uw), len(uw)); n_ = nd[b].sum()
        bs['red_mean_peak_%'].append(red(agg['pk_u'][b].sum(), agg['pk_s'][b].sum()))
        bs['red_overload_kWh_P95_%'].append(red(agg['oe_u_P95'][b].sum(), agg['oe_s_P95'][b].sum()))
        bs['red_overload_h_P95_%'].append(red(agg['oh_u_P95'][b].sum(), agg['oh_s_P95'][b].sum()))
    for k, v in bs.items():
        v = np.array(v, float); v = v[np.isfinite(v)]
        res[k.replace('%', 'lo95')] = float(np.percentile(v, 2.5)) if len(v) > 10 else np.nan
        res[k.replace('%', 'hi95')] = float(np.percentile(v, 97.5)) if len(v) > 10 else np.nan
    return res

# ---------- cache simulation inputs ----------
CTX = {}
for site in SITES:
    for per, (t0, t1) in PERIODS.items():
        days = pd.date_range(t0, t1, freq='D'); nH = len(days) * 24
        S = prep_sessions(site, t0, t1); u = uncontrolled_load(S, nH)
        ed = eval_days(site, t0, t1)
        CTX[(site, per)] = dict(S=S, nH=nH, days=days, u=u, ed=ed)
        print(f'site {site} {per}: {len(S["c"]):,} sessions | {len(days)} days | {len(ed)} evaluation days')
print('setup done')

## 3. Tune kappa on Val, evaluate on Test

In [ ]:
# ---------- peaks used to set the daily cap, per forecaster ----------
def peaks_for(forecaster, site, per, run):
    ctx = CTX[(site, per)]; ed = ctx['ed']
    if forecaster == 'Oracle (true peak)':
        U = ctx['u'][:ctx['nH']].reshape(-1, 24); d0 = ctx['days'][0]
        return {d: float(U[int((d - d0) / pd.Timedelta(days=1))].max()) for d in ed}
    if forecaster == 'Last-week peak (no model)': return lastweek_peaks(site, ed)
    return forecast_peaks(site, run, forecaster, ed)
FORECASTERS = MODELS + ['Last-week peak (no model)', 'Oracle (true peak)']

def run_sim(site, per, peaks, kappa):
    ctx = CTX[(site, per)]; capH = caps_from_peaks(peaks, ctx['days'], ctx['nH'], kappa)
    sim = simulate(ctx['S'], ctx['nH'], capH); return sim

def metrics_of(site, per, sim, rng):
    ctx = CTX[(site, per)]; st = day_stats(ctx['u'], sim['load'], ctx['nH'], ctx['ed'], ctx['days'], site)
    res = summarize(st, rng); S = ctx['S']
    res['energy_ratio'] = float(sim['load'].sum() / ctx['u'].sum()); res['sessions_violating_deadline'] = int(sim['viol'].sum())
    res['share_sessions_violating_%'] = 100.0 * float(sim['viol'].mean())
    cent_u = 0.5 * (S['c'] + S['dn']) ; cent_s = np.where(S['E'] > 0, sim['cent'] / np.maximum(S['E'], 1e-9), cent_u)
    res['mean_delay_h'] = float(np.sum(S['E'] * (cent_s - cent_u)) / S['E'].sum())
    res['shifted_kWh_per_day'] = float(0.5 * np.abs(sim['load'][:ctx['nH']] - ctx['u'][:ctx['nH']]).sum() / len(ctx['days']))
    return res

rng = np.random.default_rng(SEED)
# ---------- 1) tune kappa on Val (run1 forecasts are out-of-sample on Val) ----------
KSEL, ROWS_K = {}, []
t0_ = time.time()
for site in SITES:
    for f in FORECASTERS:
        pk = peaks_for(f, site, 'val', 'run1')
        if len(pk) == 0: continue
        best = None
        for kap in KAPPAS:
            sim = run_sim(site, 'val', pk, kap); m = metrics_of(site, 'val', sim, rng)
            ROWS_K.append(dict(site=site, forecaster=f, kappa=kap, val_mean_peak=m['mean_peak_s'], val_mean_peak_uncontrolled=m['mean_peak_u'],
                               val_red_mean_peak_pct=m['red_mean_peak_%'], val_overload_kWh_P95=m['overload_kWh_s_P95'], violations=m['sessions_violating_deadline']))
            key = (m['mean_peak_s'], m['overload_kWh_s_P95'])
            if best is None or key < best[0]: best = (key, kap)
        KSEL[(site, f)] = best[1]
    print(f'site {site}: kappa tuned for {len(FORECASTERS)} forecasters ({time.time()-t0_:.0f}s)')
KTBL = pd.DataFrame(ROWS_K); KTBL.to_csv(os.path.join(OUT_DIR, 'rq3_kappa_selection.csv'), index=False)
print(pd.Series({f'{k[0]}|{k[1]}': v for k, v in KSEL.items()}).to_string())

# ---------- 2) evaluate on Test ----------
RES_ROWS, SIMS = [], {}
for site in SITES:
    ctx = CTX[(site, 'test')]
    base = metrics_of(site, 'test', dict(load=ctx['u'].copy(), cent=0.5 * (ctx['S']['c'] + ctx['S']['dn']) * ctx['S']['E'], viol=np.zeros(len(ctx['S']['c']), bool),
                                         rem=np.zeros(len(ctx['S']['c']))), rng)
    for f in FORECASTERS:
        for run, kmode in [('run1', 'tuned'), ('run2', 'tuned'), ('run1', 'fixed')]:
            if run == 'run2' and f in ('Last-week peak (no model)', 'Oracle (true peak)'): continue
            pk = peaks_for(f, site, 'test', run)
            if len(pk) == 0: continue
            kap = KSEL.get((site, f), KAPPA_FIX) if kmode == 'tuned' else KAPPA_FIX
            sim = run_sim(site, 'test', pk, kap); m = metrics_of(site, 'test', sim, rng)
            if f in MODELS:
                a = PRED_ALL[(PRED_ALL.model == f) & (PRED_ALL.site == site) & (PRED_ALL.run == run) & (PRED_ALL.split == 'test')]
                a = a[a.time.dt.normalize().isin(ctx['ed'])]; m['forecast_MAE_H24'] = float((a.actual - a.pred).abs().mean())
            RES_ROWS.append(dict(site=site, forecaster=f, run=run, kappa_mode=kmode, kappa=kap, **m))
            SIMS[(site, f, run, kmode)] = sim
    print(f'site {site} test done ({time.time()-t0_:.0f}s)')
RES3 = pd.DataFrame(RES_ROWS)
RES3.to_csv(os.path.join(OUT_DIR, 'rq3_results.csv'), index=False)
cols = ['site', 'forecaster', 'run', 'kappa_mode', 'kappa', 'n_days', 'red_mean_peak_%', 'red_mean_peak_lo95', 'red_mean_peak_hi95', 'red_max_peak_%',
        'red_overload_h_P95_%', 'red_overload_kWh_P95_%', 'red_overload_kWh_P95_lo95', 'red_overload_kWh_P95_hi95', 'energy_ratio',
        'sessions_violating_deadline', 'mean_delay_h', 'shifted_kWh_per_day', 'forecast_MAE_H24']
print(RES3[(RES3.kappa_mode == 'tuned') & (RES3.run == 'run1')][[c for c in cols if c in RES3.columns]].round(3).to_string())
BASE = {site: dict(mean_peak=float(RES3[RES3.site == site].mean_peak_u.iloc[0]), max_peak=float(RES3[RES3.site == site].max_peak_u.iloc[0]),
                   overload_h_P95=float(RES3[RES3.site == site].overload_h_u_P95.iloc[0]), overload_kWh_P95=float(RES3[RES3.site == site].overload_kWh_u_P95.iloc[0]),
                   overload_h_P90=float(RES3[RES3.site == site].overload_h_u_P90.iloc[0]), overload_kWh_P90=float(RES3[RES3.site == site].overload_kWh_u_P90.iloc[0]),
                   n_days=int(RES3[RES3.site == site].n_days.iloc[0]), limit=LIMIT[site]) for site in SITES}
print('\nUncontrolled baseline on Test evaluation days:'); print(pd.DataFrame(BASE).T.to_string())

## 4. Figures and summary

In [ ]:
# ---------- figures ----------
MAIN = RES3[(RES3.kappa_mode == 'tuned') & (RES3.run == 'run1')]
def val_mae(site, f):
    a = PRED_ALL[(PRED_ALL.model == f) & (PRED_ALL.site == site) & (PRED_ALL.run == 'run1') & (PRED_ALL.split == 'val')]
    return (a.actual - a.pred).abs().mean() if len(a) else np.inf
BEST = {site: (min(MODELS, key=lambda f: val_mae(site, f)) if MODELS else None) for site in SITES}
print('best model by Val MAE (H=24):', BEST)
COL = {f: c for f, c in zip(FORECASTERS, plt.cm.tab10.colors)}

# F1: example week
fig, axs = plt.subplots(len(SITES), 1, figsize=(11, 3.6 * len(SITES)), squeeze=False)
for i, site in enumerate(SITES):
    ctx = CTX[(site, 'test')]; f = BEST[site]; sim = SIMS[(site, f, 'run1', 'tuned')]
    st = day_stats(ctx['u'], sim['load'], ctx['nH'], ctx['ed'], ctx['days'], site)
    # week with the highest mean uncontrolled peak among complete evaluation weeks
    wk = pd.Series(st['pk_u']).groupby(st['week']).agg(['mean', 'size']); wk = wk[wk['size'] >= 5]
    w = wk['mean'].idxmax(); d0 = ctx['days'][0] + pd.Timedelta(days=int(w) * 7); a = int((d0 - ctx['days'][0]) / pd.Timedelta(hours=1))
    t = pd.date_range(d0, periods=168, freq='h')
    ax = axs[i, 0]; ax.plot(t, ctx['u'][a:a + 168], color='#444', lw=1.4, label='Uncontrolled')
    ax.plot(t, sim['load'][a:a + 168], color='#D55E00', lw=1.4, label=f'Smart charging ({f})')
    ax.axhline(LIMIT[site]['P95'], color='#0072B2', ls='--', lw=1, label='Assumed grid limit (P95 of Train)')
    ax.set(title=f'Site {site}: example week with the highest uncontrolled peak', ylabel='Site load (kWh per hour)'); ax.legend(fontsize=7)
plt.tight_layout(); savefig('F1_example_week')

# F2: % reduction per forecaster
fig, axs = plt.subplots(len(SITES), 2, figsize=(13, 3.8 * len(SITES)), squeeze=False)
for i, site in enumerate(SITES):
    g = MAIN[MAIN.site == site].set_index('forecaster').reindex([f for f in FORECASTERS if f in set(MAIN[MAIN.site == site].forecaster)])
    for j, (col, ttl) in enumerate([('red_mean_peak_%', 'Reduction of mean daily peak (%)'), ('red_overload_kWh_P95_%', 'Reduction of overload energy above P95 limit (%)')]):
        lo = g[col] - g[col.replace('%', 'lo95')]; hi = g[col.replace('%', 'hi95')] - g[col]
        axs[i, j].barh(g.index, g[col], xerr=[lo.fillna(0), hi.fillna(0)], color=[COL[f] for f in g.index], capsize=2)
        axs[i, j].set(title=f'Site {site}: {ttl}', xlabel='% versus uncontrolled'); axs[i, j].tick_params(labelsize=7)
plt.tight_layout(); savefig('F2_reduction_by_forecaster')

# F3: accuracy vs reduction
fig, axs = plt.subplots(1, len(SITES), figsize=(5.6 * len(SITES), 4), squeeze=False)
for i, site in enumerate(SITES):
    g = MAIN[(MAIN.site == site) & MAIN.forecast_MAE_H24.notna()] if 'forecast_MAE_H24' in MAIN else MAIN.iloc[0:0]
    for _, r in g.iterrows(): axs[0, i].scatter(r.forecast_MAE_H24, r['red_mean_peak_%'], s=60, color=COL[r.forecaster]); axs[0, i].annotate(r.forecaster, (r.forecast_MAE_H24, r['red_mean_peak_%']), fontsize=6, xytext=(3, 3), textcoords='offset points')
    o = MAIN[(MAIN.site == site) & (MAIN.forecaster == 'Oracle (true peak)')]
    if len(o): axs[0, i].axhline(o['red_mean_peak_%'].iloc[0], color='k', ls=':', lw=1, label='Oracle'); axs[0, i].legend(fontsize=7)
    axs[0, i].set(title=f'Site {site}', xlabel='Day-ahead forecast MAE on Test (kWh per hour)', ylabel='Reduction of mean daily peak (%)')
plt.tight_layout(); savefig('F3_accuracy_vs_reduction')

# F4: daily peak distribution
fig, axs = plt.subplots(1, len(SITES), figsize=(5.6 * len(SITES), 3.8), squeeze=False)
for i, site in enumerate(SITES):
    ctx = CTX[(site, 'test')]; f = BEST[site]; sim = SIMS[(site, f, 'run1', 'tuned')]
    st = day_stats(ctx['u'], sim['load'], ctx['nH'], ctx['ed'], ctx['days'], site); k = st['pk_u'] > 0
    axs[0, i].boxplot([st['pk_u'][k], st['pk_s'][k]], showfliers=True); axs[0, i].set_xticklabels(['Uncontrolled', f'Smart ({f})'])
    axs[0, i].axhline(LIMIT[site]['P95'], color='#0072B2', ls='--', lw=1); axs[0, i].tick_params(axis='x', labelsize=7)
    axs[0, i].set(title=f'Site {site}: daily peak load on Test days', ylabel='kWh per hour')
plt.tight_layout(); savefig('F4_daily_peak_distribution')

# F5: kappa curves on Val
fig, axs = plt.subplots(1, len(SITES), figsize=(5.6 * len(SITES), 3.8), squeeze=False)
for i, site in enumerate(SITES):
    for f in FORECASTERS:
        g = KTBL[(KTBL.site == site) & (KTBL.forecaster == f)]
        if len(g): axs[0, i].plot(g.kappa, g.val_red_mean_peak_pct, 'o-', ms=3, color=COL[f], label=f)
    axs[0, i].set(title=f'Site {site}: kappa selection on Val', xlabel='kappa', ylabel='Reduction of mean daily peak (%)'); axs[0, i].legend(fontsize=5)
plt.tight_layout(); savefig('F5_kappa_selection')

# ---------- summary for the paper ----------
def pick(site, f, run='run1', kmode='tuned'):
    g = RES3[(RES3.site == site) & (RES3.forecaster == f) & (RES3.run == run) & (RES3.kappa_mode == kmode)]
    return g.iloc[0].to_dict() if len(g) else None
SUMMARY = {'assumptions': {'grid_limit': 'P95 of Train uncontrolled hourly load (P90 as sensitivity); no transformer rating in the data',
                           'min_session_power_kW': P_MIN_KW, 'kappa_grid': KAPPAS, 'kappa_fixed': KAPPA_FIX},
           'uncontrolled_test': BASE, 'best_model_by_val_mae': BEST, 'per_site': {}}
for site in SITES:
    d = {}
    for f in FORECASTERS:
        r = pick(site, f); r2 = pick(site, f, 'run2'); rf = pick(site, f, 'run1', 'fixed')
        if r is None: continue
        d[f] = {k: (None if (isinstance(v, float) and not np.isfinite(v)) else v) for k, v in r.items()}
        if r2: d[f]['run2_red_mean_peak_%'] = r2['red_mean_peak_%']; d[f]['run2_red_overload_kWh_P95_%'] = r2['red_overload_kWh_P95_%']
        if rf: d[f]['fixed_kappa_red_mean_peak_%'] = rf['red_mean_peak_%']; d[f]['fixed_kappa_red_overload_kWh_P95_%'] = rf['red_overload_kWh_P95_%']
    SUMMARY['per_site'][site] = d
with open(os.path.join(OUT_DIR, 'rq3_summary.json'), 'w') as fh: json.dump(SUMMARY, fh, indent=2, default=lambda o: float(o) if isinstance(o, (np.floating, np.integer)) else str(o))
print('\n================ RQ3 SUMMARY ================')
for site in SITES:
    for f in FORECASTERS:
        r = pick(site, f)
        if r: print(f'site {site} | {f:28s} kappa={r["kappa"]:.1f} | mean daily peak reduction {r["red_mean_peak_%"]:+.1f}% [{r["red_mean_peak_lo95"]:.1f},{r["red_mean_peak_hi95"]:.1f}] | '
                    f'overload kWh(P95) reduction {r["red_overload_kWh_P95_%"]:+.1f}% | violations {r["sessions_violating_deadline"]} | delay {r["mean_delay_h"]:.2f} h')
print('files:', sorted(os.listdir(OUT_DIR)))

## Paper figures (Figs. 5 and 6), drawn with seaborn/matplotlib from this notebook's own tables; PNG and CSV go to `OUT_DIR/paper_figures`.

In [ ]:
# ===== PAPER FIGURES: style chung (seaborn + matplotlib), giữ nhất quán giữa các notebook =====
import seaborn as sns
OKABE = {'blue': '#0072B2', 'orange': '#E69F00', 'green': '#009E73', 'red': '#D55E00', 'purple': '#CC79A7', 'sky': '#56B4E9', 'yellow': '#F0E442', 'grey': '#7F7F7F', 'black': '#222222'}
sns.set_theme(context='paper', style='ticks', font='DejaVu Sans', rc={
    'font.size': 7, 'axes.titlesize': 7.5, 'axes.labelsize': 7, 'xtick.labelsize': 6.5, 'ytick.labelsize': 6.5,
    'legend.fontsize': 6.5, 'legend.frameon': False, 'axes.spines.top': False, 'axes.spines.right': False,
    'axes.linewidth': 0.6, 'xtick.major.width': 0.6, 'ytick.major.width': 0.6, 'xtick.major.size': 2.5, 'ytick.major.size': 2.5,
    'lines.linewidth': 1.2, 'axes.titleweight': 'bold', 'axes.titlelocation': 'left', 'figure.dpi': 110, 'savefig.dpi': 300})
FIG_DIR = os.path.join(OUT_DIR, 'paper_figures'); os.makedirs(FIG_DIR, exist_ok=True)
def panel_tag(ax, s):                                  # nhãn (a), (b), ... ở góc trên trái, như bài báo
    ax.text(-0.02, 1.08, s, transform=ax.transAxes, fontsize=8, fontweight='bold', va='bottom', ha='right')
def save_paper_fig(fig, name):
    p = os.path.join(FIG_DIR, name + '.png'); fig.savefig(p, bbox_inches='tight', pad_inches=0.03); print('Đã lưu', p)

In [ ]:
# ===== Fig. 5 và Fig. 6 của bài báo: tính từ RES3 của notebook này =====
SHORT3 = {'Persistence (tuned)': 'Persistence', 'Seasonal Naive': 'Seasonal naive', 'Last-week peak (no model)': 'Last-week peak', 'Oracle (true peak)': 'True peak (oracle)', 'Hybrid': 'Hybrid', 'LSTM': 'LSTM'}
R1 = RES3[RES3.run == 'run1'].copy(); R1['Forecaster'] = R1.forecaster.map(SHORT3)
ORD = ['Hybrid', 'LSTM', 'Persistence', 'Seasonal naive', 'Last-week peak', 'True peak (oracle)']; ORD = [f for f in ORD if f in set(R1.Forecaster)]
KM = {'tuned': OKABE['blue'], 'fixed': OKABE['orange']}
# --- Fig. 5: PLRR của đỉnh trung bình theo ngày, khoảng tin cậy 95 % bootstrap theo tuần ---
sites_ = sorted(R1.site.unique())
fig, ax = plt.subplots(1, len(sites_), figsize=(4.8, 1.9), sharey=True, sharex=True, gridspec_kw={'wspace': 0.08})
ax = np.atleast_1d(ax)
for a, s in zip(ax, sites_):
    t = R1[(R1.site == s) & (R1.kappa_mode == 'tuned')].set_index('Forecaster').reindex(ORD); f = R1[(R1.site == s) & (R1.kappa_mode == 'fixed')].set_index('Forecaster').reindex(ORD)
    y = np.arange(len(ORD)); a.axvline(0, color='k', lw=0.5)
    a.errorbar(t['red_mean_peak_%'], y, xerr=[t['red_mean_peak_%'] - t['red_mean_peak_lo95'], t['red_mean_peak_hi95'] - t['red_mean_peak_%']], fmt='o', ms=3.5, color=KM['tuned'], elinewidth=0.9, capsize=1.5, label='Cap ratio tuned on validation')
    a.plot(f['red_mean_peak_%'], y, 'D', ms=3.5, mfc='white', mec=KM['fixed'], mew=1.0, label='Fixed cap ratio 0.8')
    a.set_yticks(y); a.set_yticklabels(ORD); a.set_title(f'Site {s}'); a.set_xlabel('PLRR of mean daily peak, %')
    a.grid(axis='x', lw=0.3, alpha=0.6)
ax[0].invert_yaxis()                                   # sharey: chỉ đảo trục một lần
h, l = ax[0].get_legend_handles_labels(); fig.legend(h, l, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.17), fontsize=6)
panel_tag(ax[0], '(a)'); panel_tag(ax[-1], '(b)')
save_paper_fig(fig, 'fig5_rq3_plrr'); plt.show()
# --- Fig. 6: giảm năng lượng vượt giới hạn lưới, giới hạn P95 (hàng trên, có khoảng tin cậy) và P90 (hàng dưới) ---
LIM = [('P95', 'red_overload_kWh_P95_%', 'Limit at P95 of training load'), ('P90', 'red_overload_kWh_P90_%', 'Limit at P90 of training load')]
fig, ax = plt.subplots(2, len(sites_), figsize=(4.8, 3.1), sharey=True, sharex=True, squeeze=False, gridspec_kw={'hspace': 0.45, 'wspace': 0.08})
for i, (lk, col, ttl) in enumerate(LIM):
    for j, s in enumerate(sites_):
        a = ax[i, j]; g = R1[R1.site == s]; y = np.arange(len(ORD)); w = 0.36
        for k, (km, off) in enumerate([('tuned', -w / 2), ('fixed', w / 2)]):
            q = g[g.kappa_mode == km].set_index('Forecaster').reindex(ORD)
            a.barh(y + off, q[col], w, color=KM[km], label='Tuned cap ratio' if km == 'tuned' else 'Fixed cap ratio 0.8')
            lo, hi = f'red_overload_kWh_{lk}_lo95', f'red_overload_kWh_{lk}_hi95'
            if lo in q.columns and q[lo].notna().any(): a.errorbar(q[col], y + off, xerr=[(q[col] - q[lo]).clip(lower=0), (q[hi] - q[col]).clip(lower=0)], fmt='none', ecolor='k', elinewidth=0.5, capsize=1)
        a.axvline(0, color='k', lw=0.5); a.set_yticks(y); a.set_yticklabels(ORD)
        if j > 0: a.tick_params(labelleft=False)
        a.set_title(f'Site {s}, {ttl.split(" at ")[1].split(" of")[0]}' if False else f'Site {s}, limit at {lk}')
        if i == 1: a.set_xlabel('Reduction of overload energy, %')
ax[0, 0].invert_yaxis()
h, l = ax[0, 0].get_legend_handles_labels(); fig.legend(h, l, loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.07), fontsize=6)
for a, t in zip(ax.ravel(), 'abcd'): panel_tag(a, f'({t})')
save_paper_fig(fig, 'fig6_rq3_overload'); plt.show()